# Data Aggregation and Group Operations

In [41]:
import numpy as np
import pandas as pd
PREVIOUS_MAX_ROWS = pd.options.display.max_rows
pd.options.display.max_rows = 20
np.random.seed(12345)
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

In this
chapter, you will learn how to:
1. Split a pandas object into pieces using one or more keys (in the form of functions,
arrays, or DataFrame column names)
2. Calculate group summary statistics, like count, mean, or standard deviation, or a
user-defined function
3. Apply within-group transformations or other manipulations, like normalization,
linear regression, rank, or subset selection
4. Compute pivot tables and cross-tabulations
5. Perform quantile analysis and other statistical group analyses

ถ้าเจอเนื้อหาที่ correspond bullet points i ให้ใส่ tag ว่า #learn_i

>Aggregation of time series data, a special use case of groupby, is
referred to as resampling in this book and will receive separate
treatment in Chapter 11.

## GroupBy Mechanics

![](chan_folder/split-apply-combine.png)

Each grouping key can take many forms, and the keys do not have to be all of the
same type:
1. A list or array of values that is the same length as the axis being grouped
2. A value indicating a column name in a DataFrame
3. A dict or Series giving a correspondence between the values on the axis being grouped and the group names
3. A function to be invoked on the axis index or the individual labels in the index


ถ้าตรงไหนใช้ key type: i ก็ให้ใส่ tag #key_i

In [42]:
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randn(5),
                   'data2' : np.random.randn(5)})
df

,key1,key2,data1,data2
0,a,one,-0.204708,1.393406
1,a,two,0.478943,0.092908
2,b,one,-0.519439,0.281746
3,b,two,-0.555730,0.769023
4,a,one,1.965781,1.246435


In [4]:
#key_1
grouped = df['data1'].groupby(df['key1'])
grouped

In [5]:
grouped.mean()

key1
a    0.746672
b   -0.537585
Name: data1, dtype: float64

Later, I’ll explain more about what happens when you call `.mean()`. The important
thing here is that the data (a Series) has been aggregated according to the group key,
producing a new Series that is now indexed by the unique values in the `key1` column

The result index has the name 'key1' because the DataFrame column `df['key1']`
did(see below.)

In [11]:
df['key1']

0    a
1    a
2    b
3    b
4    a
Name: key1, dtype: object

In [6]:
#key_1
means = df['data1'].groupby([df['key1'], df['key2']]).mean()
means
# เนื่องจากว่า keys=[a, one]|[b, one]|[b, two] ปรากฎแค่ครั้งเดียว mean ก็เท่ากับค่าใน df ดั้งเดิม

key1  key2
a     one     0.880536
      two     0.478943
b     one    -0.519439
      two    -0.555730
Name: data1, dtype: float64

In [7]:
means.unstack()

key2,one,two
key1,,
a,0.880536,0.478943
b,-0.519439,-0.555730


In [8]:
states = np.array(['Ohio', 'California', 'California', 'Ohio', 'Ohio'])
years = np.array([2005, 2005, 2006, 2005, 2006])
df['data1'].groupby([states, years]).mean()

California  2005    0.478943
            2006   -0.519439
Ohio        2005   -0.380219
            2006    1.965781
Name: data1, dtype: float64

In [9]:
#key_2
df.groupby('key1').mean()

,data1,data2
key1,,
a,0.746672,0.910916
b,-0.537585,0.525384


In [10]:
#key_2
df.groupby(['key1', 'key2']).mean()

data1     data2
key1 key2                    
a    one   0.880536  1.319920
     two   0.478943  0.092908
b    one  -0.519439  0.281746
     two  -0.555730  0.769023

You may have noticed in the first case `df.groupby('key1').mean()` that there is no
`key2` column in the result. Because `df['key2']` is not numeric data, it is said to be a
nuisance column, which is therefore excluded from the result. By default, all of the
numeric columns are aggregated, **though it is possible to filter down to a subset, as
you’ll see soon.**

ตรงที่ bold หมายถึง สามารถเลือกแค่บาง numeric columns เท่านั้นที่จะ ทำการ aggregate

In [12]:
df.groupby(['key1', 'key2']).size()

key1  key2
a     one     2
      two     1
b     one     1
      two     1
dtype: int64

### Iterating Over Groups

The GroupBy object supports iteration, generating a sequence of 2-tuples containing
the group name along with the chunk of data.

In [13]:
for name, group in df.groupby('key1'):
    print(name)
    print(group)

a
  key1 key2     data1     data2
0    a  one -0.204708  1.393406
1    a  two  0.478943  0.092908
4    a  one  1.965781  1.246435
b
  key1 key2     data1     data2
2    b  one -0.519439  0.281746
3    b  two -0.555730  0.769023


In [14]:
for (k1, k2), group in df.groupby(['key1', 'key2']):
    print((k1, k2))
    print(group)

('a', 'one')
  key1 key2     data1     data2
0    a  one -0.204708  1.393406
4    a  one  1.965781  1.246435
('a', 'two')
  key1 key2     data1     data2
1    a  two  0.478943  0.092908
('b', 'one')
  key1 key2     data1     data2
2    b  one -0.519439  0.281746
('b', 'two')
  key1 key2    data1     data2
3    b  two -0.55573  0.769023


In [15]:
# df.groupby(..) เป็น iterable ที่ element เป็น group name กับ data ถ้าแปลงเป็น list ก็จะเป็น list of 2-tuple ซึ่งสามารถ
# เอาไปสร้าง dict ได้ (เหมือนใน หมวด 3.1.4)
pieces = dict(list(df.groupby('key1')))
pieces['b']

,key1,key2,data1,data2
2,b,one,-0.519439,0.281746
3,b,two,-0.555730,0.769023


In [18]:
print(df.dtypes)
grouped = df.groupby(df.dtypes, axis=1)

key1      object
key2      object
data1    float64
data2    float64
dtype: object


In [22]:
for dtype, group in grouped:
    print(dtype)
    print(group)

float64
      data1     data2
0 -0.204708  1.393406
1  0.478943  0.092908
2 -0.519439  0.281746
3 -0.555730  0.769023
4  1.965781  1.246435
object
  key1 key2
0    a  one
1    a  two
2    b  one
3    b  two
4    a  one


#myexplain
ปกติ row ที่ ตำแหน่งที่ค่าใน column/Series ซ้ำกัน จะถูก grouped เข้าด้วยกัน โดย แต่ละ row จะเอาถูกเอามาทุกคอลัมน์

สำหรับกรณี axis=1 ดังข้างบน ตำแหน่ง 1,2 (หรือ 3,4) เป็นตำแหน่งที่ ค่าใน Series df.dtypes ซ้ำกัน ดังนั้นคอลัมน์ 1,2 (หรือ 3,4) ก็จะถูก grouped เข้าด้วยกัน โดยเวลาเอาคอลัมน์นึงมา จะเอามาทุกๆ row

### Selecting a Column or Subset of Columns

![](chan_folder/groupby_obj.png)

In [31]:
d = df.groupby('key1')
next(iter(d))
# d ไม่ใช่ dict และ 1st elemnt ของ tuple เป็น a กับ b ไม่มี data1

('a',
   key1 key2     data1     data2
 0    a  one -0.204708  1.393406
 1    a  two  0.478943  0.092908
 4    a  one  1.965781  1.246435)

```
df.groupby('key1')['data1']
df.groupby('key1')[['data2']]
```
เป้น syntactic sugar ของ
```
df['data1'].groupby(df['key1'])
df[['data2']].groupby(df['key1'])
```

In [32]:
df.groupby(['key1', 'key2'])[['data2']].mean()

data2
key1 key2          
a    one   1.319920
     two   0.092908
b    one   0.281746
     two   0.769023

#lower <br/>
The object returned by this indexing operation is a grouped DataFrame if a list or
array is passed or a grouped Series if only a single column name is passed as a scalar:

In [33]:
s_grouped = df.groupby(['key1', 'key2'])['data2']
s_grouped
s_grouped.mean()

key1  key2
a     one     1.319920
      two     0.092908
b     one     0.281746
      two     0.769023
Name: data2, dtype: float64

### Grouping with Dicts and Series

In [3]:
people = pd.DataFrame(np.random.randn(5, 5),
                      columns=['a', 'b', 'c', 'd', 'e'],
                      index=['Joe', 'Steve', 'Wes', 'Jim', 'Travis'])
people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values
people

,a,b,c,d,e
Joe,-0.204708,0.478943,-0.519439,-0.555730,1.965781
Steve,1.393406,0.092908,0.281746,0.769023,1.246435
Wes,1.007189,NaN,NaN,0.228913,1.352917
Jim,0.886429,-2.001637,-0.371843,1.669025,-0.438570
Travis,-0.539741,0.476985,3.248944,-1.021228,-0.577087


In [4]:
mapping = {'a': 'red', 'b': 'red', 'c': 'blue',
           'd': 'blue', 'e': 'red', 'f' : 'orange'}
# column 'a' corresponds to red, 'b' to red, 'c' to blue, ...

In [5]:
by_column = people.groupby(mapping, axis=1)
by_column.sum()

,blue,red
Joe,-1.075169,2.240016
Steve,1.050769,2.732748
Wes,0.228913,2.360106
Jim,1.297183,-1.553778
Travis,2.227716,-0.639844


In [6]:
#retained
map_series = pd.Series(mapping)
map_series
people.groupby(map_series, axis=1).count()
# เนื่องจากเรา group columns, count is perform across column, และ rows are retained ทำให้ จน. row เท่ากับใน orig. data frame
# Because of missing values in row 'Wes'

,blue,red
Joe,2,3
Steve,2,3
Wes,1,2
Jim,2,3
Travis,2,3


In [18]:
#ลองเอง
for color, group in people.groupby(map_series, axis=1):
    print(color)
    print(group.join(group.count(axis=1).rename('count')), '\n')

blue
               c         d  count
Joe    -0.519439 -0.555730      2
Steve   0.281746  0.769023      2
Wes          NaN  0.228913      1
Jim    -0.371843  1.669025      2
Travis  3.248944 -1.021228      2 

red
               a         b         e  count
Joe    -0.204708  0.478943  1.965781      3
Steve   1.393406  0.092908  1.246435      3
Wes     1.007189       NaN  1.352917      2
Jim     0.886429 -2.001637 -0.438570      3
Travis -0.539741  0.476985 -0.577087      3 



### Grouping with Functions

Using Python functions is a more generic way of defining a group mapping compared
with a dict or Series. **Any function passed as a group key will be called once per index
value, with the return values being used as the group names**. More concretely, consider
the example DataFrame from the previous section, which has people’s first
names as index values. Suppose you wanted to group by the length of the names;
while you could compute an array of string lengths, it’s simpler to just pass the `len`
function:

In [19]:
people.groupby(len).sum()

,a,b,c,d,e
3,1.688911,-1.522694,-0.891281,1.342208,2.880128
5,1.393406,0.092908,0.281746,0.769023,1.246435
6,-0.539741,0.476985,3.248944,-1.021228,-0.577087


In [33]:
#ลองเอง # same result
ser = people.index.map(len).to_series()
ser.index = people.index
print(ser,'\n')

people.groupby(ser).sum()

Joe       3
Steve     5
Wes       3
Jim       3
Travis    6
dtype: int64 



,a,b,c,d,e
3,1.688911,-1.522694,-0.891281,1.342208,2.880128
5,1.393406,0.092908,0.281746,0.769023,1.246435
6,-0.539741,0.476985,3.248944,-1.021228,-0.577087


Mixing functions with arrays, dicts, or Series is not a problem as everything gets converted
to arrays internally:

In [34]:
key_list = ['one', 'one', 'one', 'two', 'two']
people.groupby([len, key_list]).min()

a         b         c         d         e
3 one -0.204708  0.478943 -0.519439 -0.555730  1.352917
  two  0.886429 -2.001637 -0.371843  1.669025 -0.438570
5 one  1.393406  0.092908  0.281746  0.769023  1.246435
6 two -0.539741  0.476985  3.248944 -1.021228 -0.577087

### Grouping by Index Levels

In [35]:
columns = pd.MultiIndex.from_arrays([['US', 'US', 'US', 'JP', 'JP'],
                                    [1, 3, 5, 1, 3]],
                                    names=['cty', 'tenor'])
hier_df = pd.DataFrame(np.random.randn(4, 5), columns=columns)
hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0      0.124121  0.302614  0.523772  0.000940  1.343810
1     -0.713544 -0.831154 -2.370232 -1.860761 -0.860757
2      0.560145 -1.265934  0.119827 -1.063512  0.332883
3     -2.359419 -0.199543 -1.541996 -0.970736 -1.307030

In [36]:
#retained
hier_df.groupby(level='cty', axis=1).count()
# computation across columns, rows are retained
# คำตอบควรมี 4 แถว สองคอลัมน์(US, JP)

cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


In [37]:
#ลองเอง #bigpicture
hier_df.groupby(level='cty', axis=1).count().columns.name
## ในกรณีนี้ (axis=1)
# ค่าของ Column-index ซ้ำๆกันได้ กลายมาเป็น column-index ทีมี unique value
# ชื่อของ column-index กลายมาเป็น ชื่อของ column-index (ถ้าสำหรับข้างบน ชื่อของ column กลายมาเป็นชื่อของ index)
## ในกรณีที่ผ่านมาด้านบนๆ (axis=0)
# column ที่ใช้เป็น key ที่มีค่าซ้ำๆ กัน กลายมาเป็น row-index ที่มี unique value

'cty'

## Data Aggregation

While `quantile` is not explicitly implemented for GroupBy, it is a Series method and
thus available for use

In [44]:
print(df)
grouped = df.groupby('key1')
grouped['data1'].quantile(0.9)
# ที่ออกมาเป็น series เพราะ subset ด้วย single value โดยไม่มี bracket ซ้อนอีกชั้น

  key1 key2     data1     data2
0    a  one -0.204708  1.393406
1    a  two  0.478943  0.092908
2    b  one -0.519439  0.281746
3    b  two -0.555730  0.769023
4    a  one  1.965781  1.246435


key1
a    1.668413
b   -0.523068
Name: data1, dtype: float64

In [45]:
#ลองเอง 
# columns are retained ทำทีละกรุป กรุปที่ key1 == 'a' ทำทุกๆคอลัมน์ ต่อมา กรุปที่ key1 == 'b' ทำทุกๆคอลัมน์
grouped.quantile(0.9)

,data1,data2
key1,,
a,1.668413,1.364012
b,-0.523068,0.720295


In [46]:
def peak_to_peak(arr):
    return arr.max() - arr.min()
grouped.agg(peak_to_peak)

,data1,data2
key1,,
a,2.170488,1.300498
b,0.036292,0.487276


In [58]:
#crosscheck
print(df, '\n')
# for key1=='a'
df_a = df.loc[[0, 1, 4], ['data1', 'data2']].apply(peak_to_peak)
# for key1=='b'
df_b = df.loc[[2, 3], ['data1', 'data2']].apply(peak_to_peak)

pd.DataFrame([df_a, df_b])

  key1 key2     data1     data2
0    a  one -0.204708  1.393406
1    a  two  0.478943  0.092908
2    b  one -0.519439  0.281746
3    b  two -0.555730  0.769023
4    a  one  1.965781  1.246435 



,data1,data2
0,2.170488,1.300498
1,0.036292,0.487276


In [ ]:
grouped.describe()

### Column-Wise and Multiple Function Application

In [ ]:
tips = pd.read_csv('examples/tips.csv')
# Add tip percentage of total bill
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips[:6]

In [ ]:
grouped = tips.groupby(['day', 'smoker'])

In [ ]:
grouped_pct = grouped['tip_pct']
grouped_pct.agg('mean')

In [ ]:
grouped_pct.agg(['mean', 'std', peak_to_peak])

In [ ]:
grouped_pct.agg([('foo', 'mean'), ('bar', np.std)])

In [ ]:
functions = ['count', 'mean', 'max']
result = grouped['tip_pct', 'total_bill'].agg(functions)
result

In [ ]:
result['tip_pct']

In [ ]:
ftuples = [('Durchschnitt', 'mean'), ('Abweichung', np.var)]
grouped['tip_pct', 'total_bill'].agg(ftuples)

In [ ]:
grouped.agg({'tip' : np.max, 'size' : 'sum'})
grouped.agg({'tip_pct' : ['min', 'max', 'mean', 'std'],
             'size' : 'sum'})

### Returning Aggregated Data Without Row Indexes

In [ ]:
tips.groupby(['day', 'smoker'], as_index=False).mean()

## Apply: General split-apply-combine

In [ ]:
def top(df, n=5, column='tip_pct'):
    return df.sort_values(by=column)[-n:]
top(tips, n=6)

In [ ]:
tips.groupby('smoker').apply(top)

In [ ]:
tips.groupby(['smoker', 'day']).apply(top, n=1, column='total_bill')

In [ ]:
result = tips.groupby('smoker')['tip_pct'].describe()
result
result.unstack('smoker')

f = lambda x: x.describe()
grouped.apply(f)

### Suppressing the Group Keys

In [ ]:
tips.groupby('smoker', group_keys=False).apply(top)

### Quantile and Bucket Analysis

In [ ]:
frame = pd.DataFrame({'data1': np.random.randn(1000),
                      'data2': np.random.randn(1000)})
quartiles = pd.cut(frame.data1, 4)
quartiles[:10]

In [ ]:
def get_stats(group):
    return {'min': group.min(), 'max': group.max(),
            'count': group.count(), 'mean': group.mean()}
grouped = frame.data2.groupby(quartiles)
grouped.apply(get_stats).unstack()

In [ ]:
# Return quantile numbers
grouping = pd.qcut(frame.data1, 10, labels=False)
grouped = frame.data2.groupby(grouping)
grouped.apply(get_stats).unstack()

### Example: Filling Missing Values with Group-Specific       Values

In [ ]:
s = pd.Series(np.random.randn(6))
s[::2] = np.nan
s
s.fillna(s.mean())

In [ ]:
states = ['Ohio', 'New York', 'Vermont', 'Florida',
          'Oregon', 'Nevada', 'California', 'Idaho']
group_key = ['East'] * 4 + ['West'] * 4
data = pd.Series(np.random.randn(8), index=states)
data

In [ ]:
data[['Vermont', 'Nevada', 'Idaho']] = np.nan
data
data.groupby(group_key).mean()

In [ ]:
fill_mean = lambda g: g.fillna(g.mean())
data.groupby(group_key).apply(fill_mean)

In [ ]:
fill_values = {'East': 0.5, 'West': -1}
fill_func = lambda g: g.fillna(fill_values[g.name])
data.groupby(group_key).apply(fill_func)

### Example: Random Sampling and Permutation

In [ ]:
# Hearts, Spades, Clubs, Diamonds
suits = ['H', 'S', 'C', 'D']
card_val = (list(range(1, 11)) + [10] * 3) * 4
base_names = ['A'] + list(range(2, 11)) + ['J', 'K', 'Q']
cards = []
for suit in ['H', 'S', 'C', 'D']:
    cards.extend(str(num) + suit for num in base_names)

deck = pd.Series(card_val, index=cards)

In [ ]:
deck[:13]

In [ ]:
def draw(deck, n=5):
    return deck.sample(n)
draw(deck)

In [ ]:
get_suit = lambda card: card[-1] # last letter is suit
deck.groupby(get_suit).apply(draw, n=2)

In [ ]:
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

### Example: Group Weighted Average and Correlation

In [ ]:
df = pd.DataFrame({'category': ['a', 'a', 'a', 'a',
                                'b', 'b', 'b', 'b'],
                   'data': np.random.randn(8),
                   'weights': np.random.rand(8)})
df

In [ ]:
grouped = df.groupby('category')
get_wavg = lambda g: np.average(g['data'], weights=g['weights'])
grouped.apply(get_wavg)

In [ ]:
close_px = pd.read_csv('examples/stock_px_2.csv', parse_dates=True,
                       index_col=0)
close_px.info()
close_px[-4:]

In [ ]:
spx_corr = lambda x: x.corrwith(x['SPX'])

In [ ]:
rets = close_px.pct_change().dropna()

In [ ]:
get_year = lambda x: x.year
by_year = rets.groupby(get_year)
by_year.apply(spx_corr)

In [ ]:
by_year.apply(lambda g: g['AAPL'].corr(g['MSFT']))

### Example: Group-Wise Linear Regression

In [ ]:
import statsmodels.api as sm
def regress(data, yvar, xvars):
    Y = data[yvar]
    X = data[xvars]
    X['intercept'] = 1.
    result = sm.OLS(Y, X).fit()
    return result.params

In [ ]:
by_year.apply(regress, 'AAPL', ['SPX'])

## Pivot Tables and Cross-Tabulation

In [ ]:
tips.pivot_table(index=['day', 'smoker'])

In [ ]:
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],
                 columns='smoker')

In [ ]:
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],
                 columns='smoker', margins=True)

In [ ]:
tips.pivot_table('tip_pct', index=['time', 'smoker'], columns='day',
                 aggfunc=len, margins=True)

In [ ]:
tips.pivot_table('tip_pct', index=['time', 'size', 'smoker'],
                 columns='day', aggfunc='mean', fill_value=0)

### Cross-Tabulations: Crosstab

In [ ]:
from io import StringIO
data = """\
Sample  Nationality  Handedness
1   USA  Right-handed
2   Japan    Left-handed
3   USA  Right-handed
4   Japan    Right-handed
5   Japan    Left-handed
6   Japan    Right-handed
7   USA  Right-handed
8   USA  Left-handed
9   Japan    Right-handed
10  USA  Right-handed"""
data = pd.read_table(StringIO(data), sep='\s+')

In [ ]:
data

In [ ]:
pd.crosstab(data.Nationality, data.Handedness, margins=True)

In [ ]:
pd.crosstab([tips.time, tips.day], tips.smoker, margins=True)

In [ ]:
pd.options.display.max_rows = PREVIOUS_MAX_ROWS

## Conclusion